In [ ]:
import torch
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
import torch
from torch import nn
import numpy as np
from torch.optim import Adam, SGD, RMSprop, AdamW
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR
import time
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker 
from networks import DCGANDiscriminator, DCGANGenerator, init_weights
from datasets import create_dataset
from sampling import *
from evaluation import evaluate

In [ ]:
lr_G= 0.0005
lr_D= 0.0005
batch_size = 128
num_epochs = 50
latent_dim = 128
k = 1

In [ ]:
def get_optimizer(optimizer_name, parameters, lr):
    if optimizer_name == "adam":
        return Adam(parameters, lr=lr, betas=(0.5, 0.999))
    elif optimizer_name == "sgd":
        return SGD(parameters, lr=lr, momentum=0.9)
    elif optimizer_name == "rmsprop":
        return RMSprop(parameters, lr=lr)
    elif optimizer_name == "adamw":
        return AdamW(parameters, lr=lr)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

In [ ]:
def get_scheduler(scheduler_name, optimizer, num_epochs, steps_per_epoch):
    if scheduler_name == "none":
        return None
    elif scheduler_name == "step":
        return StepLR(optimizer, step_size=10, gamma=0.5)
    elif scheduler_name == "cosine":
       return CosineAnnealingLR(optimizer, T_max=steps_per_epoch * num_epochs, eta_min=1e-5)

    else:
        raise ValueError(f"Unknown scheduler: {scheduler_name}")

In [ ]:
def train(generator, discriminator, loader, show_generator_output=True, visualize_training_const=False,
          optimizer_type="adam", scheduler_type="none"):

  
    #device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    #if device.type == 'cuda':
        #print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    #else:
        #print("Using CPU")
    #print("=========================================")

    if torch.cuda.is_available():
        device = torch.device("cuda:0")
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")
    else:
        device = torch.device("cpu")
        print("Using CPU")

    print("=========================================")

    generator.to(device)
    discriminator.to(device)
    print(f"Start training DCGAN! Training for {num_epochs} epochs.")

    training_loader, test_loader, dims, img_size = loader
    steps_per_epoch = len(training_loader)

    optimizer_D = get_optimizer(optimizer_type, discriminator.parameters(), lr_D)
    optimizer_G = get_optimizer(optimizer_type, generator.parameters(), lr_G)

    scheduler_D = get_scheduler(scheduler_type, optimizer_D, num_epochs, steps_per_epoch)
    scheduler_G = get_scheduler(scheduler_type, optimizer_G, num_epochs, steps_per_epoch)

    bce_loss = nn.BCEWithLogitsLoss()

    images = []
    const_z = torch.randn(5, latent_dim, 1, 1, device=device)
    
    losses_D = []
    losses_G = []
    test_losses_D = []
    test_losses_G = []

    learning_rates = []  # Für Learning Rate Plot

     # Für Komponentenplot (alle 150 Batches)
    d_real_losses = []
    d_fake_losses = []
    g_losses_per_batch = []
    
    

    if visualize_training_const:
        images = visualize_latent_space(generator, device, const_z, dims, img_size)

    for epoch in range(num_epochs):
        epoch_start = time.time()

        for batch_idx, (real_imgs, _) in enumerate(training_loader):
            real_imgs = real_imgs.to(device)
        
            real_labels = torch.full((real_imgs.shape[0], 1), 1, device=device)
            fake_labels = torch.zeros((real_imgs.shape[0], 1), device=device)

            # --- Discriminator ---
            for _ in range(k):
                z = torch.randn(real_imgs.shape[0], latent_dim, 1, 1, device=device)
                fake_imgs = generator(z)
                
                real_loss = bce_loss(discriminator(real_imgs), real_labels)
                fake_loss = bce_loss(discriminator(fake_imgs.detach()), fake_labels)
                d_loss = 0.9 * real_loss + fake_loss

                optimizer_D.zero_grad()
                d_loss.backward()
                optimizer_D.step()

            # --- Generator ---
            z = torch.randn(real_imgs.shape[0], latent_dim, 1, 1, device=device)
            generated_imgs = generator(z)
            g_loss = bce_loss(discriminator(generated_imgs), real_labels)

            optimizer_G.zero_grad()
            g_loss.backward()
            optimizer_G.step()
            
            if batch_idx % 150 == 0:
                print(f"Batch {batch_idx}: D Loss = {d_loss.item():.4f} | G Loss = {g_loss.item():.4f} | LR = {optimizer_D.param_groups[0]['lr']:.6f}")
                
                # Logging für Komponenten-Plot
                d_real_losses.append(real_loss.item())
                d_fake_losses.append(fake_loss.item())
                g_losses_per_batch.append(g_loss.item())
               
                # Logging für LR-Plot
                learning_rates.append(optimizer_D.param_groups[0]['lr'])
                

        losses_D.append(d_loss.item())
        losses_G.append(g_loss.item())

        test_d_loss, test_g_loss = evaluate(discriminator, generator, latent_dim, test_loader)
        test_losses_D.append(test_d_loss)
        test_losses_G.append(test_g_loss)

        # --- Step schedulers after optimizer update ---
        if scheduler_D: scheduler_D.step()
        if scheduler_G: scheduler_G.step()

       
        print(f"Epoch [{epoch+1}/{num_epochs}] | D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f} | Test D: {test_d_loss:.4f} | Test G: {test_g_loss:.4f} | Time: {time.time() - epoch_start:.2f}s")

        if show_generator_output:
            if img_size == 64:
                generate_sampels_celebA(generator, device, latent_dim, 5)
            elif dims == 3:
                generate_sampels_cifar10(generator, device, latent_dim, 5)
            else:
                generate_sampels_mnist(generator, device, latent_dim, 5)

        if visualize_training_const:
            images = torch.cat((images, visualize_latent_space(generator, device, const_z, dims, img_size)), dim=0)

    if visualize_training_const:
        save_plot(images, filename=f"{optimizer_type}_{scheduler_type}_output", image_scale=2, device=device)

    return losses_D, losses_G, test_losses_D, test_losses_G, learning_rates, d_real_losses, d_fake_losses, g_losses_per_batch


In [ ]:

lr_D = 0.00005
lr_G = 0.0001
batch_size = 128
num_epochs = 50
latent_dim = 128
discriminator_minst = DCGANDiscriminator(1, 32, 32)
generator_minst = DCGANGenerator(latent_dim, 1, 32, 32)

results_minst = train(
    generator_minst,
    discriminator_minst,
    create_dataset("mnist"),
    optimizer_type="adam",
    scheduler_type="step",
    show_generator_output=True,
    visualize_training_const=True 
)

(
    losses_D_minst,
    losses_G_minst,
    test_D_minst,
    test_G_minst,
    learning_rates_minst,
    d_real_losses_minst,
    d_fake_losses_minst,
    g_losses_per_batch_minst
) = results_minst

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].plot(losses_D_minst, label="Train D Loss")
axs[0].plot(losses_G_minst, label="Train G Loss")
axs[0].scatter(range(len(test_D_minst)), test_D_minst, label="Test D Loss", color='red')
axs[0].scatter(range(len(test_G_minst)), test_G_minst, label="Test G Loss", color='green')
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Loss")
axs[0].legend()
axs[0].set_title("Training & Test Losses")
axs[0].grid()

axs[1].plot(learning_rates_minst, label="Learning Rate")
axs[1].set_xlabel("Logged Batch (every 150th)")
axs[1].set_ylabel("Learning Rate")
axs[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2e'))
axs[1].set_title("Learning Rate Over Time")
axs[1].grid()
axs[1].legend()

axs[2].plot(d_real_losses_minst, label="D Real Loss")
axs[2].plot(d_fake_losses_minst, label="D Fake Loss")
axs[2].plot(g_losses_per_batch_minst, label="G Loss")
axs[2].set_xlabel("Logged Batch (every 150th)")
axs[2].set_ylabel("Loss")
axs[2].set_title("Loss Components")
axs[2].grid()
axs[2].legend()

plt.tight_layout()
plt.show()

In [ ]:

lr_D = 0.0005
lr_G = 0.0005

batch_size = 128
num_epochs = 50
latent_dim = 128

discriminator_minst_2 = DCGANDiscriminator(1, 32, 32)
generator_minst_2 = DCGANGenerator(latent_dim, 1, 32, 32)


results_minst = train(
    generator_minst_2,
    discriminator_minst_2,
    create_dataset("mnist"),
    optimizer_type="adamw",
    scheduler_type="step",
    show_generator_output=True,
    visualize_training_const=True 
)

(
    losses_D_minst,
    losses_G_minst,
    test_D_minst,
    test_G_minst,
    learning_rates_minst,
    d_real_losses_minst,
    d_fake_losses_minst,
    g_losses_per_batch_minst
) = results_minst

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].plot(losses_D_minst, label="Train D Loss")
axs[0].plot(losses_G_minst, label="Train G Loss")
axs[0].scatter(range(len(test_D_minst)), test_D_minst, label="Test D Loss", color='red')
axs[0].scatter(range(len(test_G_minst)), test_G_minst, label="Test G Loss", color='green')
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Loss")
axs[0].legend()
axs[0].set_title("Training & Test Losses")
axs[0].grid()

axs[1].plot(learning_rates_minst, label="Learning Rate")
axs[1].set_xlabel("Logged Batch (every 150th)")
axs[1].set_ylabel("Learning Rate")
axs[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2e'))
axs[1].set_title("Learning Rate Over Time")
axs[1].grid()
axs[1].legend()

axs[2].plot(d_real_losses_minst, label="D Real Loss")
axs[2].plot(d_fake_losses_minst, label="D Fake Loss")
axs[2].plot(g_losses_per_batch_minst, label="G Loss")
axs[2].set_xlabel("Logged Batch (every 150th)")
axs[2].set_ylabel("Loss")
axs[2].set_title("Loss Components")
axs[2].grid()
axs[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
lr_G = 0.0002
lr_D = 0.0002

batch_size = 128
num_epochs = 100
latent_dim = 128

discriminator_cfar10 = DCGANDiscriminator(3,64,32)
generator_cfar10 = DCGANGenerator(latent_dim,3,64,32)

results_cfar10 = train(
    generator_cfar10,
    discriminator_cfar10,
    create_dataset("cifar10"),
    optimizer_type="adam",
    scheduler_type="step",
    show_generator_output=True,
    visualize_training_const=True
)

( 
    losses_D_cfar10, 
    losses_G_cfar10, 
    test_D_cfar10, 
    test_G_cfar10, 
    learning_rates_cfar10, 
    d_real_losses_cfar10, 
    d_fake_losses_cfar10, 
    g_losses_per_batch_cfar10   
) = results_cfar10

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].plot(losses_D_cfar10, label="Train D Loss")
axs[0].plot(losses_G_cfar10, label="Train G Loss")
axs[0].scatter(range(len(test_D_cfar10)), test_D_cfar10, label="Test D Loss", color='red')
axs[0].scatter(range(len(test_G_cfar10)), test_G_cfar10, label="Test G Loss", color='green')
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Loss")
axs[0].legend()
axs[0].set_title("Training & Test Losses")
axs[0].grid()

axs[1].plot(learning_rates_cfar10, label="Learning Rate")
axs[1].set_xlabel("Logged Batch (every 150th)")
axs[1].set_ylabel("Learning Rate")
axs[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2e'))
axs[1].set_title("Learning Rate Over Time")
axs[1].grid()
axs[1].legend()

axs[2].plot(d_real_losses_cfar10, label="D Real Loss")
axs[2].plot(d_fake_losses_cfar10, label="D Fake Loss")
axs[2].plot(g_losses_per_batch_cfar10, label="G Loss")
axs[2].set_xlabel("Logged Batch (every 150th)")
axs[2].set_ylabel("Loss")
axs[2].set_title("Loss Components")
axs[2].grid()
axs[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
lr_G = 0.0002
lr_D = 0.0001

batch_size = 128
num_epochs = 100
latent_dim = 128

discriminator_cifar10_2 = DCGANDiscriminator(3, 64, 32)
generator_cifar10_2 = DCGANGenerator(latent_dim, 3, 64, 32)

generator_cifar10_2.apply(init_weights)
discriminator_cifar10_2.apply(init_weights)


results_cifar10 = train(
    generator_cifar10_2,
    discriminator_cifar10_2,
    create_dataset("cifar10"),
    optimizer_type="adam",
    scheduler_type="cosine",
    show_generator_output=True,
    visualize_training_const=False
)
(
    losses_D_cifar10,
    losses_G_cifar10,
    test_D_cifar10,
    test_G_cifar10,
    learning_rates_cifar10,
    d_real_losses_cifar10,
    d_fake_losses_cifar10,
    g_losses_per_batch_cifar10
) = results_cifar10

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].plot(losses_D_cifar10, label="Train D Loss")
axs[0].plot(losses_G_cifar10, label="Train G Loss")
axs[0].scatter(range(len(test_D_cfar10)), test_D_cifar10, label="Test D Loss", color='red')
axs[0].scatter(range(len(test_G_cfar10)), test_G_cifar10, label="Test G Loss", color='green')
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Loss")
axs[0].legend()
axs[0].set_title("Training & Test Losses")
axs[0].grid()

axs[1].plot(learning_rates_cifar10, label="Learning Rate")
axs[1].set_xlabel("Logged Batch (every 150th)")
axs[1].set_ylabel("Learning Rate")
axs[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2e'))
axs[1].set_title("Learning Rate Over Time")
axs[1].grid()
axs[1].legend()

axs[2].plot(d_real_losses_cifar10, label="D Real Loss")
axs[2].plot(d_fake_losses_cifar10, label="D Fake Loss")
axs[2].plot(g_losses_per_batch_cifar10, label="G Loss")
axs[2].set_xlabel("Logged Batch (every 150th)")
axs[2].set_ylabel("Loss")
axs[2].set_title("Loss Components")
axs[2].grid()
axs[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
%pip install gdown

In [ ]:
discriminator_celeba = DCGANDiscriminator(3,64,64).to(device)
generator_celeba = DCGANGenerator(latent_dim,3,64,64).to(device)

lr_G = 0.0002
lr_D = 0.0001

batch_size = 128
num_epochs = 100
latent_dim = 128


generator_celeba.apply(init_weights)
discriminator_celeba.apply(init_weights)


results_celeba = train(
    generator_celeba,
    discriminator_celeba,
    create_dataset("celeba"),
    optimizer_type="adam",
    scheduler_type="cosine",
    show_generator_output=True,
    visualize_training_const=True,

)
losses_D_celeba, losses_G_celeba, test_D_celeba, test_G_celeba, learning_rates_celeba, d_real_losses_celeba, d_fake_losses_celeba, g_losses_per_batch_celeba = results_celeba

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].plot(losses_D_celeba, label="Train D Loss")
axs[0].plot(losses_G_celeba, label="Train G Loss")
axs[0].scatter(range(len(test_D_celeba)), test_D_celeba, label="Test D Loss", color='red')
axs[0].scatter(range(len(test_G_celeba)), test_G_celeba, label="Test G Loss", color='green')
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Loss")
axs[0].legend()
axs[0].set_title("Training & Test Losses")
axs[0].grid()

axs[1].plot(learning_rates_celeba, label="Learning Rate")
axs[1].set_xlabel("Logged Batch (every 150th)")
axs[1].set_ylabel("Learning Rate")
axs[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2e'))
axs[1].set_title("Learning Rate Over Time")
axs[1].grid()
axs[1].legend()

axs[2].plot(d_real_losses_celeba, label="D Real Loss")
axs[2].plot(d_fake_losses_celeba, label="D Fake Loss")
axs[2].plot(g_losses_per_batch_celeba, label="G Loss")
axs[2].set_xlabel("Logged Batch (every 150th)")
axs[2].set_ylabel("Loss")
axs[2].set_title("Loss Components")
axs[2].grid()
axs[2].legend()
plt.tight_layout()
plt.show()